In [2]:
# SoRL in modded-gpt compatible fashion (for ultra fast pre-training)
# 1. pre-training demands simple model architecture, even .generate function can be wrapped around the trained model afterwards
# 2. no need to include 'kv-cache' for the pre-training experiment here

In [ ]:
from sorl.model import CausalSelfAttention, Block, GPTConfig
import torch 

# mock input 
x = torch.randn(2, 1024, 768)

config = GPTConfig()
attn = CausalSelfAttention(dim=768, n_head=6)
block = Block(config=config)
y, v1 = attn(x)
x, v1 =block(x, v1, x, None)

In [7]:
import torch 
from sorl.gat import GATConfig, GAT

gat_config = GATConfig(vocab_sizes=[128,8],
          n_layer=12,
          n_head=6,
          n_embd=768,
          flex_kernel_options=None)

model = GAT(gat_config)


token_ids = torch.randint(0, 128 + 8, (2, 4))
idx = token_ids[:, :-1].contiguous()
target = token_ids[:, 1:].contiguous()


# forward pass 
ppt = model(idx, target, 1024)

# denoise 
denoise_mask = torch.tensor([[False, True, True, False], [False, True, True, True]])
assert denoise_mask[:, 0].sum() == 0, "first token should not be denoised"
denoised_logits = model.denoise(token_ids, denoise_mask, 1024)


In [10]:
from sorl.gat import parallel_denoise
from sorl.gat import generate 


num_iterations = 5 
memory_span = 1024 
temperature = 0.0

parallel_denoise(model, idx, num_iterations=5, memory_span=1024, temperature=0.0)

generate(model, idx, max_new_tokens=5, abstraction_interval=3)


tensor([[ 18,  13, 102, 129,   0,   0, 129,   0],
        [ 92,  97,  56, 129,   0,   0, 129,   0]])